# **Assignment Two**
**Student Name:** Jessica Mawuenam Dellason

**Student Number:** 25215734

This project will be based on [Review Files]("http://mlg.ucd.ie/modules/python/reviews.zip")

## **AI Declaration Statement**
How to use with to open two files at a time 

Understanding #oc with some user_id; anonymous but they are still users maybe specisal users cause their id start with R

Converting html to str 

Removing special characters in title


In [52]:
# Import necessary libraries
import json
import pandas as pd
from bs4 import BeautifulSoup
import re
import matplotlib.pyplot as plt
import seaborn as sns

---
## **Task 1: Data Preparation and Characterisation**

In [53]:
# Load 'review-scores.json' and 'reviews/review-text.json' file into a DataFrame
try:
    with open("reviews/review-scores.json", 'r') as scores_file, open("reviews/review-text.json", 'r') as text_file:
        scores_df = pd.DataFrame(json.loads(scores_file.read()))
        text_df = pd.DataFrame(json.loads(text_file.read()))

except FileNotFoundError:
    print("Either 'reviews/review-scores.json' or 'reviews/review-text.json' file not found")

# Merge 'scores_df' and 'text_df' on review_id
reviews_df = text_df.merge(scores_df, on = "review_id")

# Save merged data into csv file
reviews_df.to_csv("reviews.csv", index= False)

In [54]:
# Check for missing cells
reviews_df.isna().sum()

review_id         0
product_id        0
user_id           0
review_title     45
review_body      45
review_date       0
rating            0
helpful_votes     0
total_votes       0
dtype: int64

### **Inspecting Review ID**
All `review_id` values start with 'R' and are unique. This column was used as the index for the dataset.

In [55]:
# Verify that every review_id includes "R" (returns True if all do)
print(f"All 'review_id' start with 'R': {reviews_df[reviews_df["review_id"].str.startswith("R")].shape[0] == reviews_df.shape[0]}")

# Verify that each review_id is unique
print(f"Each 'review_id' is unique: {reviews_df["review_id"].nunique() == reviews_df.shape[0]}")

# Set 'review_id' as index
reviews_df.set_index('review_id', inplace=True)

All 'review_id' start with 'R': True
Each 'review_id' is unique: True


### **Inspecting Product ID**
All `product_id` values start with 'B', except for 6 entries that were purely numeric, likely due to a scraping error. The prefix 'B' was added to these 6 values to maintain consistency. 

In [56]:
# Verify that every product_id includes "B" (returns True if all do)
print(f"All 'product_id' conatain 'B': {reviews_df[reviews_df["product_id"].str.contains("B")].shape[0] == reviews_df.shape[0]}")

# Get the reviews whose product_id does not contain 'B'
print(f"All 'product_id' not conataining 'B': {reviews_df[~reviews_df["product_id"].str.contains("B")].shape[0]}")

# Iterate over rows where product_id does NOT contain "B" and attach B to the beginning
for index, row in reviews_df[~reviews_df["product_id"].str.contains("B")].iterrows():
    reviews_df.loc[index, "product_id"] = 'B' + str(row["product_id"])

# Verify all rows
print(f"All 'product_id' conatain 'B': {reviews_df[reviews_df["product_id"].str.contains("B")].shape[0] == reviews_df.shape[0]}")

All 'product_id' conatain 'B': False
All 'product_id' not conataining 'B': 6
All 'product_id' conatain 'B': True


### **Inspecting User ID**
All `user_id` values start with 'A', except for 24 that started with `#oc-R`, likely representing anonymous or special reviewers.  
It seems that:  
- `A` → regular users  
- `R` → special users (from `#oc-R`)  

The `#oc-` prefix was removed, keeping the rest of the user ID intact for consistency.

In [57]:
# Verify that every user_id starts with "A" (returns True if all do)
print(f"All 'user_id' starts with 'A': {reviews_df[reviews_df["user_id"].str.startswith("A")].shape[0] == reviews_df.shape[0]}")

# Get the reviews whose user_id does not start with 'A'
print(f"All 'user_id' not starting with 'A': {reviews_df[~reviews_df["user_id"].str.startswith("A")].shape[0]}")

# View the dataframe
print(f"Dataframe of user_id not starting with A:\n{reviews_df[~reviews_df["user_id"].str.startswith("A")].iloc[:,0:2]}")
 
# Remove "#oc-" prefix from all user_id values
reviews_df["user_id"] = reviews_df["user_id"].str.replace("#oc-", "", regex=False)

All 'user_id' starts with 'A': False
All 'user_id' not starting with 'A': 24
Dataframe of user_id not starting with A:
           product_id             user_id
review_id                                
R136454    B006Q820X0  #oc-R14ZUK54VMOGJS
R136363    B006Q820X0   #oc-RV1CL5MB39LEK
R136470    B006Q820X0  #oc-R3376HJIKZWS95
R516118    B008I1XPKA  #oc-R1AQH0RQMTL1Y9
R516116    B008I1XPKA   #oc-R3HMF6ODAC2R4
R212349    B005EF0I0I  #oc-R2ZTU0FEJBQXE7
R136446    B006Q820X0  #oc-R11T1PHWNO7KEZ
R136455    B006Q820X0  #oc-R152UR09M996EM
R136429    B006Q820X0  #oc-R2GDRKIV15IZHW
R136430    B006Q820X0  #oc-R2O4E9SPR08RNA
R83654     B005ZBZLT4  #oc-R2I896V302UXAQ
R181219    B007Y59HVM   #oc-RPS46SRDKTRZI
R124841    B005EF0HRM  #oc-R3NZKZGLJEKBN3
R181208    B007Y59HVM  #oc-R2I896V302UXAQ
R136487    B006Q820X0  #oc-R1GUQD6SLYACOQ
R124838    B005EF0HRM  #oc-R2ULR70UT6GZPQ
R136416    B006Q820X0  #oc-R2CIG5JYXHVPEE
R83668     B005ZBZLT4  #oc-R2C7F72WD2NEUW
R541159    B005EF0HTK  #oc-R1ZR5L29T4LSAE

### **Inspecting Rating**
The ratings are represented using `*` (stars).  
For analysis, these have been converted to their numeric equivalents (e.g., `****` -> 4) to allow calculations and statistical summaries.


In [58]:
# Convert star ratings (e.g. "***") to numeric values (3)
reviews_df["rating"] = reviews_df["rating"].str.len()

### **Inspecting Review Date**
The `review_date` column was converted to a proper date type for analysis.  
Since the dataset contained multiple date formats, `format='mixed'` was used to handle the different formats automatically.

In [59]:
# Change 'review_date' to date format
reviews_df["review_date"] = pd.to_datetime(reviews_df["review_date"], format='mixed')

### **Inspecting Review Body**
The `review_body` column contained HTML tags such as `<br>` and `<p>`.  
A function was applied to convert the HTML content to plain text using `BeautifulSoup`.  
Non-string values (e.g., missing entries) were replaced with an empty string to avoid errors.

In [60]:
def html_to_text(html):
    if not isinstance(html, str):
        return ""
    return BeautifulSoup(html, "html.parser").get_text(separator=" ")

# Overwrite the existing column
reviews_df["review_body"] = reviews_df["review_body"].apply(html_to_text)

### **Inspecting Review Title**
The `review_title` column was cleaned and standardized in three steps:  

1. **Remove whitespace** - Leading and trailing spaces were stripped.  
2. **Format text** - Converted to title case for consistency.  
3. **Remove special characters** - All non-alphanumeric characters were removed, while keeping spaces.  
Non-string values were left unchanged to avoid errors.

In [61]:
# Remove whitespace
reviews_df["review_title"] = reviews_df["review_title"].str.strip()

# Format text 
reviews_df["review_title"] = reviews_df["review_title"].str.title()

# Remove special characters
reviews_df["review_title"] = reviews_df["review_title"].apply(
    lambda x: re.sub(r"[^a-zA-Z0-9\s]", "", x) if isinstance(x, str) else x
)

### **Dataset Overview**
- **Total reviews:** 15,375  
- **Columns:** 8 (product_id, user_id, review_title, review_body, review_date, rating, helpful_votes, total_votes)  
- **Index:** review_id (unique)  
- **Missing values:** review_title (45 missing) , review_body (45 missing) 
- **Review dates:** From 2021-01-01 to 2025-12-30  
- **product_id:** 6,331 unique, most frequent appears 90 times  
- **user_id:** 9,786 unique, most frequent appears 104 times  

In [62]:
# Overview of dataset
print(reviews_df.info()) 

# Summary statistics for numerical features
print(reviews_df.describe())

# Summary for categorical/text features
print(reviews_df[["product_id", "user_id"]].describe(include="object"))

# Look at first few rows
reviews_df.head()

<class 'pandas.DataFrame'>
Index: 15375 entries, R274002 to R327437
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   product_id     15375 non-null  str           
 1   user_id        15375 non-null  str           
 2   review_title   15330 non-null  str           
 3   review_body    15375 non-null  str           
 4   review_date    15375 non-null  datetime64[us]
 5   rating         15375 non-null  int64         
 6   helpful_votes  15375 non-null  int64         
 7   total_votes    15375 non-null  int64         
dtypes: datetime64[us](1), int64(3), str(4)
memory usage: 1.6+ MB
None
                      review_date        rating  helpful_votes   total_votes
count                       15375  15375.000000   15375.000000  15375.000000
mean   2023-06-28 04:38:54.907317      3.414504      21.933268     27.129041
min           2021-01-01 00:00:00      1.000000       0.000000     12.000000
25%        

/var/folders/07/s8x8wk454bn62q42l_skqgdw0000gn/T/ipykernel_45177/2090246771.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(reviews_df[["product_id", "user_id"]].describe(include="object"))


,product_id,user_id,review_title,review_body,review_date,rating,helpful_votes,total_votes
review_id,,,,,,,,
R274002,B001O1Q0NA,AY6QZ7AY3AK8Z,Fluoridated Salt,I thought this salt was fluoride free unfortun...,2021-01-01,1,17,24
R302264,B000MOEUMS,A33DXT3AD5DHCB,Not As Advertised,This product is NOT drinking chocolate. I real...,2021-01-01,3,6,12
R84546,B002ZOIKMY,A14396UCW3Z0KI,Best Bags For Iced Tea Excellent Deal,"First, Luzianne makes the best classic brewed ...",2021-01-01,5,18,18
R117515,B0016B7Z32,A1ODU9IV6O87IB,DonT Pay For Expedited Delivery,"I paid over $13.00 for ""expedited"" delivery an...",2021-01-01,2,4,22
R131840,B001XUO8AY,A3LOCNXB7Z1WCH,I Love FeverTree,After doing an Internet search for tonic water...,2021-01-01,5,16,16


In [ ]:
# Distribution of reviews by year
reviews_per_year = reviews_df.groupby(reviews_df['review_date'].dt.year).size()
print(f"The reviews per year:\n{reviews_per_year}")

The reviews per year:
review_date
2021    3094
2022    3095
2023    3118
2024    3027
2025    3041
dtype: int64
